# Fine-tuned Qwen 2.5-3B-Instruct Inference for Privacy QA

This notebook performs inference using a fine-tuned Qwen 2.5-3B-Instruct model on the Privacy QA classification task.

## 1. Setup and Model Loading

In [ ]:
# Install required packages if needed
# !pip install transformers torch accelerate peft

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import pandas as pd
import re
import logging
from datetime import datetime
import json
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Configuration
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # Base model from Hugging Face
ADAPTER_PATH = "/path/to/your/qwen/adapter"  # Update this path to your fine-tuned adapter
TEST_DATA_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/nlp/privacyQA/data/PrivacyQA_EMNLP/sampled_data/stratified_test_sample_1000.csv"
OUTPUT_DIR = "/home/smaniyar_umass_edu/BioNLP_Ontology/nlp/privacyQA/test_results/finetuned_qwen/"
LOG_FILE = OUTPUT_DIR + "inference_log.log"
PREDICTIONS_FILE = OUTPUT_DIR + "predictions.csv"

# Create output directory if it doesn't exist
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Setup logging
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode='w'
)
logging.info("Starting Qwen 2.5 fine-tuned model inference...")
print(f"Logging to: {LOG_FILE}")

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

In [ ]:
# Load base model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Base model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# Load fine-tuned adapter (uncomment when you have the adapter path)
# print("Loading fine-tuned adapter...")
# model = PeftModel.from_pretrained(model, ADAPTER_PATH)
# model = model.merge_and_unload()  # Merge adapter weights
# print("Fine-tuned adapter loaded and merged.")

# For now, we'll use the base model
print("Using base model (update ADAPTER_PATH to use fine-tuned version)")

## 2. Data Loading and Preprocessing

In [ ]:
# Load test data
print("Loading test data...")
df = pd.read_csv(TEST_DATA_PATH)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Check label distribution
print("Label distribution:")
print(df['Any_Relevant'].value_counts())
print(f"\nPercentages:")
print(df['Any_Relevant'].value_counts(normalize=True) * 100)

## 3. Prompt Engineering and Inference Functions

In [ ]:
def create_prompt(query, segment):
    """Create a structured prompt for the privacy QA classification task."""
    prompt = f"""You are a legal expert classifier that determines if a segment of text from a document is relevant to a user query.

Query: "{query}"
Segment: "{segment}"

Decide whether the segment is RELEVANT or IRRELEVANT to the query.

Respond in JSON format:
{{
  "reasoning": "Your reasoning here in one or two sentences",
  "final_label": "Relevant" or "Irrelevant"
}}"""
    return prompt

# Test the prompt function
sample_query = "Do you share my data with third parties?"
sample_segment = "We may share your information with trusted partners for marketing purposes."
test_prompt = create_prompt(sample_query, sample_segment)
print("Sample prompt:")
print(test_prompt)

In [ ]:
def generate_response(prompt, max_length=512, temperature=0.7, top_p=0.9):
    """Generate response using the fine-tuned model."""
    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
        padding=True
    ).to(model.device)
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test the generation function
print("Testing generation function...")
test_response = generate_response(test_prompt)
print(f"Test response: {test_response}")

In [ ]:
def extract_reasoning_and_label(output_text):
    """Extract reasoning and label from model output."""
    output_text = output_text.strip()
    
    # Try to find JSON structure
    json_match = re.search(r'\{.*\}', output_text, re.DOTALL)
    if json_match:
        json_str = json_match.group(0)
        try:
            data = json.loads(json_str)
            reasoning = data.get("reasoning", "")
            final_label = data.get("final_label", "")
            return reasoning, final_label
        except json.JSONDecodeError:
            pass
    
    # Fallback: regex extraction
    reasoning_match = re.search(r'"?reasoning"?\s*:\s*"([^"]+)"', output_text)
    label_match = re.search(r'"?final_label"?\s*:\s*"([^"]+)"', output_text)
    
    reasoning = reasoning_match.group(1) if reasoning_match else ""
    final_label = label_match.group(1) if label_match else ""
    
    # If still no label found, look for RELEVANT/IRRELEVANT in text
    if not final_label:
        relevant_match = re.search(r'\b(RELEVANT|IRRELEVANT)\b', output_text, re.IGNORECASE)
        if relevant_match:
            final_label = relevant_match.group(1).capitalize()
    
    return reasoning, final_label

# Test extraction function
reasoning, label = extract_reasoning_and_label(test_response)
print(f"Extracted reasoning: {reasoning}")
print(f"Extracted label: {label}")

## 4. Batch Inference

In [ ]:
# Prepare results storage
results = []
df['id'] = df['DocID'].astype(str) + '_' + df['QueryID'].astype(str) + '_' + df['SentID'].astype(str)

print(f"Starting inference on {len(df)} samples...")
logging.info(f"Starting batch inference on {len(df)} samples")

# Process each sample
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing samples"):
    query = str(row['Query'])
    segment = str(row['Segment'])
    sample_id = row['id']
    ground_truth = row['Any_Relevant']
    
    # Create prompt
    prompt = create_prompt(query, segment)
    
    try:
        # Generate response
        raw_output = generate_response(prompt)
        
        # Extract reasoning and label
        reasoning, final_label = extract_reasoning_and_label(raw_output)
        
        # Log the interaction
        logging.info(
            f"[{sample_id}] Query: {query[:100]}...\n"
            f"Segment: {segment[:100]}...\n"
            f"Raw Output: {raw_output}\n"
            f"Extracted Label: {final_label}\n"
            f"Ground Truth: {ground_truth}\n"
            f"{'-'*80}"
        )
        
    except Exception as e:
        logging.error(f"[{sample_id}] Error during inference: {str(e)}")
        raw_output = "ERROR"
        reasoning = "Error occurred during inference"
        final_label = "Error"
    
    # Store results
    results.append({
        'id': sample_id,
        'Query': query,
        'Segment': segment,
        'Any_Relevant': ground_truth,
        'model_output': raw_output,
        'reasoning': reasoning,
        'final_label': final_label
    })
    
    # Print progress every 50 samples
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(df)} samples")

print("\nInference completed!")
logging.info("Batch inference completed")

## 5. Save Results

In [ ]:
# Convert results to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv(PREDICTIONS_FILE, index=False)

print(f"Results saved to: {PREDICTIONS_FILE}")
print(f"Total samples processed: {len(results_df)}")

# Display first few results
print("\nFirst 5 results:")
display_cols = ['id', 'Any_Relevant', 'final_label', 'reasoning']
print(results_df[display_cols].head())

logging.info(f"Results saved to {PREDICTIONS_FILE}")
logging.info(f"Total samples processed: {len(results_df)}")

## 6. Quick Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Normalize labels for evaluation
def normalize_label(label):
    if not isinstance(label, str):
        return None
    label = label.strip().lower()
    if "relevant" in label:
        if "irrelevant" in label:
            return 0  # Irrelevant
        return 1  # Relevant
    return None

# Prepare labels for evaluation
results_df['gold'] = results_df['Any_Relevant'].apply(lambda x: 1 if str(x).strip().lower() == "relevant" else 0)
results_df['pred'] = results_df['final_label'].apply(normalize_label)

# Filter out samples with missing predictions
eval_df = results_df.dropna(subset=['pred'])
print(f"Evaluating on {len(eval_df)} samples (dropped {len(results_df) - len(eval_df)} with missing predictions)")

if len(eval_df) > 0:
    # Calculate metrics
    accuracy = accuracy_score(eval_df['gold'], eval_df['pred'])
    f1 = f1_score(eval_df['gold'], eval_df['pred'])
    cm = confusion_matrix(eval_df['gold'], eval_df['pred'])
    
    print("\n" + "="*50)
    print("🔍 Fine-tuned Qwen 2.5 Privacy QA Evaluation")
    print("="*50)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(eval_df['gold'], eval_df['pred'], target_names=["Irrelevant", "Relevant"]))
    
    # Log evaluation results
    logging.info(f"Evaluation Results - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
    logging.info(f"Confusion Matrix: {cm.tolist()}")
else:
    print("No valid predictions found for evaluation!")
    logging.warning("No valid predictions found for evaluation")

## 7. Sample Predictions Analysis

In [ ]:
# Show some example predictions
if len(eval_df) > 0:
    print("\n" + "="*60)
    print("SAMPLE PREDICTIONS")
    print("="*60)
    
    # Correct predictions
    correct_preds = eval_df[eval_df['gold'] == eval_df['pred']]
    if len(correct_preds) > 0:
        print("\n✅ CORRECT PREDICTIONS (sample):")
        for idx, row in correct_preds.head(3).iterrows():
            print(f"\nQuery: {row['Query'][:100]}...")
            print(f"Segment: {row['Segment'][:100]}...")
            print(f"Ground Truth: {row['Any_Relevant']} | Predicted: {row['final_label']}")
            print(f"Reasoning: {row['reasoning']}")
            print("-" * 40)
    
    # Incorrect predictions
    incorrect_preds = eval_df[eval_df['gold'] != eval_df['pred']]
    if len(incorrect_preds) > 0:
        print("\n❌ INCORRECT PREDICTIONS (sample):")
        for idx, row in incorrect_preds.head(3).iterrows():
            print(f"\nQuery: {row['Query'][:100]}...")
            print(f"Segment: {row['Segment'][:100]}...")
            print(f"Ground Truth: {row['Any_Relevant']} | Predicted: {row['final_label']}")
            print(f"Reasoning: {row['reasoning']}")
            print("-" * 40)
    
    print(f"\nCorrect predictions: {len(correct_preds)}/{len(eval_df)} ({len(correct_preds)/len(eval_df)*100:.1f}%)")
    print(f"Incorrect predictions: {len(incorrect_preds)}/{len(eval_df)} ({len(incorrect_preds)/len(eval_df)*100:.1f}%)")

## 8. Summary

This notebook demonstrates inference using a fine-tuned Qwen 2.5-3B-Instruct model for Privacy QA classification.

**Key Components:**
- Model loading with adapter support
- Structured prompt engineering
- Batch inference with progress tracking
- JSON response parsing
- Comprehensive evaluation metrics
- Detailed logging and error handling

**Next Steps:**
1. Update `ADAPTER_PATH` with your fine-tuned model path
2. Adjust generation parameters for optimal performance
3. Compare results with zero-shot and other fine-tuned models
4. Analyze error patterns for further model improvements